In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.optimize
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.special import logsumexp
import warnings
from scipy.optimize import curve_fit
from scipy.optimize import approx_fprime
import scipy.stats as stats
from pybads import BADS

from matplotlib.colors import LinearSegmentedColormap

# TEMP
warnings.simplefilter("ignore", category=FutureWarning)

%cd ..
from src.src_copy import *

# Set some defaults
plt.rc("axes.spines", top=False, right=False)
plt.rcParams["mathtext.default"] = "regular"
sns.set_theme(font_scale=1.2)
sns.set_style("ticks")
sns.set_palette('deep')

%config InlineBackend.figure_format = "retina"

# Read in data to be fit
%cd "modeling"
df = pd.read_csv("kim2025.csv")

# Add columns to 2025 data that are present in error detection df
df["session"] = "adapt"
# df["no_fit"] = False
df = df.rename(columns={
    "motor_sd":"sigma_motor_adapt",
    "rotation":"delta_xv",
    "theta_maxradv":"x_f",
    "fbi":"vis_fb"
    })

# To match the column names to the 2025 data
df_err_det = pd.read_csv("combined-data.csv")
df_err_det.columns

df

__Notes:__
- ~~Set `no_fit` to `False` for all trials.~~
    - Using outlier removal function.
- There was a size mismatch in all log-prob functions between `no-fit` and `log-p` lists/arrays. Since `no-fit` is set to false for all trials, I truncated it to size match with log-p.

In [ ]:
# Remove outliers
df_clean = remove_outliers(3.5,
                                df,
                                "x_f",
                                "SN",
                                "adapt",
                                abs_cutoff=22.5,
                                switch_outlier_with_mean=False
)

# Count number of no_fit trials per participant
df_clean.groupby("SN")["no_fit"].sum()

In [ ]:
fit_single_subj = False

numtrials_adapt = 1800
numtrials_sdt = 800
numtrials_both = numtrials_adapt + numtrials_sdt

# We switch off the printing from BADS
options = {"display" : 'off'}

fit_task = "adapt"
n_sims = 1000
n_inits = 1

if fit_single_subj:
    # Pick your subject here
    subjects = np.array([14])
else:
    subjects = df_clean['SN'].unique()

In [ ]:
%%time
# PIECE parameter bounds: alpha, beta, sigma_p, sigma_pert, b,
bounds       = ((0.1, 5), (0.001, 1), (0.1, 10), (0.1, 120), (-5, 5))
plaus_bounds = ((0.5, 4), (0.01, 0.5), (0.3, 6), (3, 20), (-2.5, 2.5))

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_piece, aic, bic = fit_bads("piece", s, df_clean, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, fit_task=fit_task, options=options)
    results_list.append(fits_piece)
    aic_list.append(aic)
    bic_list.append(bic)

best_params_piece = np.array([res.x for res in results_list]).squeeze()

fits_piece = pd.DataFrame({
    'SN': subjects,
    'model': "piece",
    "fit_task": fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
fits_piece[["alpha", "beta", "sigma_p", "sigma_pert", "b"]] = pd.DataFrame(fits_piece['params'].tolist(), index=fits_piece.index)

if fit_task == "adapt":
    fits_piece_adapt = fits_piece.copy()
elif fit_task == "both":
    fits_piece_both = fits_piece.copy()
elif fit_task == "sdt": 
    fits_piece_sdt = fits_piece.copy()

# # Checks path and creates new csv
# save_csv(fits_piece, model='piece')
# fits_piece.to_csv("model-fits/nolapse_piece_simult.csv", index=False)
# fits_piece.to_csv("model-fits/nolapse_piece_adapt.csv", index=False)

fits_piece.to_csv("model-fits/fits-piece-2025-data.csv", index=False)

In [ ]:
%%time
# Fit VCO
if fit_task == "sdt":
    # Parameter bounds (C, sigma_v, b) 
    bounds       = [(0.1, 10), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.5, 6), (-2.5, 2.5)]
else:
    # Parameter bounds (C, B, sigma_v, b) 
    bounds       = [(0.1, 10), (0.01, 1), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.05, 0.6), (0.5, 6), (-2.5, 2.5)]

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_vco, aic, bic = fit_bads("vco", s, df_clean, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, 
                                  fit_task=fit_task, options=options)
    results_list.append(fits_vco)
    aic_list.append(aic)
    bic_list.append(bic)

best_params_vco = np.array([res.x for res in results_list]).squeeze()

fits_vco = pd.DataFrame({
    'SN': subjects,
    'model': "vco",
    "fit_task": fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
if fit_task == "adapt":
    fits_vco[["C", "B", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_adapt = fits_vco.copy()
elif fit_task == "both":
    fits_vco[["C", "B", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_both = fits_vco.copy()
elif fit_task == "sdt": 
    fits_vco[["C", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_sdt = fits_vco.copy()

# # Checks path and creates new csv
# fits_vco.to_csv("model-fits/SAVE_vco_sdt.csv", index=False)
# save_csv(fits_vco, model='vco')
# fits_vco.to_csv("model-fits/fixed_vco_simult.csv", index=False)
# fits_vco.to_csv("model-fits/fixed_vco_adapt.csv", index=False)

fits_vco.to_csv("model-fits/fits-vco-2025-data.csv", index=False)

In [ ]:
%%time
# Fit CO model
if fit_task == "sdt":
    # Parameter bounds (C, sigma_comb) 
    bounds = [(0.1, 10), (0.1, 10.0)]
    plaus_bounds = [(0.5, 5), (0.5, 6)]
else:
    # Parameter bounds (C, B, sigma_comb, b) 
    bounds       = [(0.1, 10), (0.01, 1), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.05, 0.6), (0.5, 6), (-2.5, 2.5)]

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_co, aic, bic = fit_bads("co", s, df_clean, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, fit_task=fit_task, options=options)
    results_list.append(fits_co)
    aic_list.append(aic)
    bic_list.append(bic)

fits_co = pd.DataFrame({
    'SN': subjects,
    'model': "co",
    "fit_task" : fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
if fit_task == "adapt":  
    fits_co[["C", "B", "sigma_comb", "b"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_adapt = fits_co.copy()
elif fit_task == "both":
    fits_co[["C", "B", "sigma_comb", "b"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_both = fits_co.copy()
elif fit_task == "sdt": 
    fits_co[["C", "sigma_comb"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_sdt = fits_co.copy()

# # Checks path and creates new csv
# save_csv(fits_co, model='co')
# fits_co.to_csv("model-fits/nolapse_co_simult.csv", index=False)
# fits_co.to_csv("model-fits/nolapse_co_adapt.csv", index=False)


fits_co.to_csv("model-fits/fits-co-2025-data.csv", index=False)

In [ ]:
df_fits = pd.concat([
    pd.read_csv("model-fits/fits-piece-2025-data.csv"),
    pd.read_csv("model-fits/fits-vco-2025-data.csv"),
    pd.read_csv("model-fits/fits-co-2025-data.csv"),
])

aic_diff = df_fits.pivot(index="SN", columns="model", values="aic").assign(
    vco=lambda x: x["piece"] - x["vco"],
    co=lambda x: x["piece"] - x["co"]
)

aic_diff[["vco", "co"]].plot(kind="bar", ylabel="AIC diff")

In [ ]:
# Melt to match code from the analyze-models nb
aic_diff_long = (
    aic_diff.
    reset_index()
    .melt(
    id_vars="SN",
    value_vars=["vco", "co"],
    var_name="model",
    value_name="aic_diff_piece"
    )       
)
aic_diff_long

In [ ]:
# fits_adapt = pd.read_csv("model-fits/fits_adapt.csv", converters={"params": converter})
fits_adapt = aic_diff_long

# Compare models on adaptation task
idx_vco = (fits_adapt["model"] == "vco") 
idx_co = (fits_adapt["model"] == "co") 
idx_piece = (fits_adapt["model"] == "piece") 

# Assign colors to models
piece_color = '#984ea3'  # purple
vco_color = '#4daf4a'  # green
co_color = '#ff7f00'  # orange

# Compare models on adaptation
fig, ax = plt.subplots()
sns.stripplot(data=fits_adapt, x="model", y="aic_diff_piece", hue="model", ax=ax, alpha=0.5, order=["vco", "co"])
ax.axhline(color="k", linewidth=1.0)
ax.set(xlabel="Model", xticklabels=["VCO", "CO"], ylabel="$\Delta$ AIC", ylim=[-400, 1000])
ax.set_title("Adaptation fits")
sns.despine()
plt.show()

colors1 = [piece_color if x >= 0 else vco_color for x in fits_adapt.loc[idx_vco, "aic_diff_piece"] * -1]
colors2 = [piece_color if x >= 0 else vco_color for x in fits_adapt.loc[idx_co, "aic_diff_piece"] * -1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True, sharey=True)
ax1.bar(x=fits_adapt.loc[idx_vco, "SN"].values, height=fits_adapt.loc[idx_vco, "aic_diff_piece"].values * -1, 
       color=colors1)
ax1.axhline(c="k", linewidth=0.8)
ax2.bar(x=fits_adapt.loc[idx_co, "SN"].values, height=fits_adapt.loc[idx_co, "aic_diff_piece"].values * -1,
        color=co_color)
ax1.set(xticks=np.arange(1, 17), ylabel="$\Delta$ AIC: VCO - PIECE", xlabel="Participant")
ax2.set(ylabel="$\Delta$ AIC: CO - PIECE", xlabel="Participant")
ax2.tick_params(labelleft=True)
ax2.axhline(c="k", linewidth=0.8)
fig.suptitle("Adaptation Task")
sns.despine()
plt.tight_layout()
fig.savefig("../figs/aic_adapt_2025_data.pdf", dpi=300)